In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 15 — Reference coding solutions

Three exercises. Read the lesson and workbook first. Learner functions intentionally raise `NotImplementedError` until you write them. Reference execution is not your assessment.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/shape_lab").is_dir())
if str(ROOT/"src") not in sys.path:sys.path.insert(0,str(ROOT/"src"))


## 15.C1 — `stereo_depth`

Compute camera-axis depth with the principal-point offset. Nonfinite disparity or a nonpositive denominator yields NaN. f and B are known positive scalars.

**Before coding:** work a small example on paper.

**Hint:** Use Z=fB/(d+offset); do not replace invalid data with zero.

In [3]:
def stereo_depth(disparity, focal, baseline, offset):
    d=np.asarray(disparity,float);den=d+offset
    valid=np.isfinite(d)&(den>0)
    z=np.full(d.shape,np.nan)
    np.divide(focal*baseline,den,out=z,where=valid)
    return z

In [4]:
z=stereo_depth([50,np.nan,np.inf,-20],1000,.2,20)
assert np.isclose(z[0],200/70)
assert np.isnan(z[1:]).all()
from shape_lab.stereo import reference_sample
_,_,d,c=reference_sample();z=stereo_depth(d,c['focal_px'],c['baseline_m'],c['doffs_px'])
assert z.shape==d.shape and np.all(z[np.isfinite(z)]>0)
print("15.C1: checks passed")

15.C1: checks passed


**Why this works and where it stops:** The output unit follows the baseline. Offset is in pixels and is added to disparity before division.

## 15.C2 — `error_and_coverage`

For matching-shaped arrays, report MAE over finite paired values and coverage relative to finite reference values. Return None when a denominator is absent.

**Before coding:** work a small example on paper.

**Hint:** An accurate result on very few points must not hide missing coverage.

In [5]:
def error_and_coverage(predicted, reference):
    p=np.asarray(predicted,float);r=np.asarray(reference,float)
    if p.shape!=r.shape:raise ValueError("Shapes must match.")
    eligible=np.isfinite(r);valid=eligible&np.isfinite(p)
    return {"mae":float(np.mean(abs(p[valid]-r[valid]))) if valid.any() else None,
            "coverage":float(valid.sum()/eligible.sum()) if eligible.any() else None}

In [6]:
assert error_and_coverage([1,np.nan,5],[2,3,np.nan])=={'mae':1.,'coverage':.5}
assert error_and_coverage([np.nan],[np.nan])=={'mae':None,'coverage':None}
print("15.C2: checks passed")

15.C2: checks passed


**Why this works and where it stops:** Coverage is not evaluated count divided by all array entries when some references are themselves missing.

## 15.C3 — `conditional_depth_sd`

Compute the first-order depth standard deviation when only disparity is uncertain and f, B, offset are fixed.

**Before coding:** work a small example on paper.

**Hint:** Differentiate fB/(d+offset), then multiply the derivative magnitude by σd.

In [7]:
def conditional_depth_sd(disparity, focal, baseline, offset, disparity_sd):
    if disparity+offset<=0 or disparity_sd<0:raise ValueError("Positive denominator and nonnegative SD required.")
    return focal*baseline/(disparity+offset)**2*disparity_sd

In [8]:
assert np.isclose(conditional_depth_sd(60,1000,.2,20,.2),.00625)
assert conditional_depth_sd(60,1000,.2,20,0)==0
assert conditional_depth_sd(20,1000,.2,20,.2)>conditional_depth_sd(60,1000,.2,20,.2)
print("15.C3: checks passed")

15.C3: checks passed


**Why this works and where it stops:** This is conditional local uncertainty, not total sensor or model uncertainty. It can fail near degeneracy or with substantial noise.